# Этап 14: ASVD (Activation-aware Singular Value Decomposition)

В этапе 6 мы делали обычное SVD. Но обзор (Раздел 6) указывает на метод **ASVD** (arXiv:2312.05821), который работает значительно лучше.

### Проблема обычного SVD:
SVD минимизирует ошибку восстановления матрицы весов $W$. Но нас не волнует сама матрица $W$, нас волнует результат $Y = WX$. Ошибка в весах, которые умножаются на маленькие активации, почти не важна.

### Идея ASVD:
Мы делаем SVD не для матрицы $W$, а для «взвешенной» матрицы $W' = W \cdot diag(X_{rms})$. 
Это заставляет SVD сохранять именно те направления, которые наиболее важны для реальных данных, проходящих через сеть.

In [ ]:
import torch
import torch.nn as nn
from src.model import GPTLanguageModel, device, get_batch
import matplotlib.pyplot as plt

model = GPTLanguageModel().to(device)
model.load_state_dict(torch.load('model_ckpt.pt', map_location=device))
model.eval()

def apply_asvd_decomposition(layer, input_data, rank=64):
    W = layer.weight.data
    
    # 1. Считаем RMS активаций для каждого входного канала
    # input_data: (B, T, C)
    x_rms = torch.sqrt(torch.mean(input_data**2, dim=(0, 1)))
    
    # 2. Масштабируем веса перед SVD
    W_scaled = W * x_rms.view(1, -1)
    
    # 3. Делаем SVD взвешенной матрицы
    U, S, Vh = torch.linalg.svd(W_scaled, full_matrices=False)
    
    # 4. Усекаем до нужного ранга и возвращаем масштабирование
    # Чтобы получить итоговые матрицы A и B, нам нужно убрать x_rms
    U_k = U[:, :rank]
    S_k = S[:rank]
    Vh_k = Vh[:rank, :]
    
    # Финальные матрицы для SVDLinear слоев:
    # B = U_k * sqrt(S_k)
    # A = (sqrt(S_k) * Vh_k) / x_rms
    mat_B = U_k * torch.sqrt(S_k)
    mat_A = (torch.sqrt(S_k).view(-1, 1) * Vh_k) / (x_rms.view(1, -1) + 1e-6)
    
    return mat_A, mat_B

target_layer = model.blocks[0].ffwd.net[0]
xb, yb = get_batch('val')

with torch.no_grad():
    # Просимулируем входные данные
    input_features = torch.randn(64, 256, 384).to(device)
    A, B = apply_asvd_decomposition(target_layer, input_features, rank=64)

print(f"Матрица A: {A.shape}, Матрица B: {B.shape}")
print("ASVD успешно вычислил компоненты, учитывая важность активаций!")